In [1]:
import psycopg2
import toytree
import toyplot
from multiprocessing import Pool

ModuleNotFoundError: No module named 'toytree'

In [9]:
# Add family size information to the DataFrame, pull this from the database
DB = {
    "host":     "petadex.ccz9y6yshbls.us-east-1.rds.amazonaws.com",
    "port":     5432,
    "database": "petadex",
    "user":     "readonly_user",
    "password": "petadex",
}

conn = psycopg2.connect(**DB)
cur  = conn.cursor()

In [9]:
cur.execute("""SELECT DISTINCT family, COUNT(*)
            FROM enzyme_taxonomy
            WHERE family IS NOT NULL
            GROUP BY family
            """)
families = cur.fetchall()

In [3]:
OUTPUT_DIR = "./data"

In [ ]:
from asyncio import subprocess


for family in families:
    family_id = family[0]
    cur.execute("""SELECT enzyme_fastaa.enzyme_id, translated_sequence
                FROM enzyme_taxonomy
                JOIN enzyme_fastaa ON enzyme_taxonomy.enzyme_id = enzyme_fastaa.enzyme_id
                WHERE family = %s
                """, (family_id,))
    rows = cur.fetchall()

    # Write the sequences to FASTA
    fasta_path = f"{OUTPUT_DIR}/{family_id}.fasta"
    with open(fasta_path, "w") as f:
        for seq_id, seq in rows:
            f.write(f">{seq_id}\n{seq}\n")

    # Run MAFFT (MSA)
    aligned_fasta = f"{OUTPUT_DIR}/{family_id}_aligned.fasta"
    subprocess.run(
        f"mafft --auto {fasta_path} > {aligned_fasta}", 
        shell=True, check=True, stderr=subprocess.DEVNULL
    )

    tree_path = f"{OUTPUT_DIR}/{family_id}.tree"
    subprocess.run(
        f"fasttree -lg < {aligned_fasta} > {tree_path}", 
        shell=True, check=True, stderr=subprocess.DEVNULL
    )
    tree_image_path = f"{OUTPUT_DIR}/{family_id}.svg"
    tree = toytree.tree(tree_path)
    canvas = toyplot.Canvas(
        width=800, 
        height=600, 
        style={"background-color": "#fdf6e3"} # Solarized Light background

    )
    axes = canvas.cartesian(label=f"Enzyme Family: {family_id}")
    axes.show = False # Hide the X/Y axis lines
    tree.draw(axes=axes, tip_labels=True, tip_labels_style={"font-size": 5})
    toyplot.svg.render(canvas, tree_image_path)
    

In [ ]:
import toytree
import toyplot.svg
import os

output_path = f"./trees/{family_id}.svg"

tree = toytree.tree("./data/1.tree")
# show tree data parsed from Newick str
canvas, axes, mark = tree.draw(
            width=800, 
            height=600, 
            node_labels=tree.get_node_data("support"),
            tip_labels=True
        )
os.makedirs(os.path.dirname(output_path), exist_ok=True)
toyplot.svg.render(canvas, output_path)        

Rendered: ./trees/63372.svg


In [ ]:
from ete3 import Tree  # Note: No TreeStyle imported here

def get_root_representative(tree_path):
    if not os.path.exists(tree_path):
        return None

    try:
        # Load the tree (Format 1 is standard for FastTree)
        t = Tree(tree_path, format=1)
        
        # 1. Perform Midpoint Rooting
        # This mathematically places the root at the center of the tree
        outgroup = t.get_midpoint_outgroup()
        t.set_outgroup(outgroup)
        
        # 2. Find the 'center' leaf
        # We want the leaf node closest to our new root
        leaves = t.get_leaves()
        representative = min(leaves, key=lambda x: x.get_distance(t))
        
        return representative.name
    except Exception as e:
        print(f"Error processing {tree_path}: {e}")
        return None

# Test it
print(get_root_representative("./data/1.tree"))

44534


In [ ]:
import toytree
import toyplot.svg

def draw_tree_with_background(tree_path, output_path, family_id):
    tre = toytree.tree(tree_path)
    canvas = toyplot.Canvas(
        width=800, 
        height=600, 
        style={"background-color": "#fdf6e3"} # Solarized Light background

    )
    axes = canvas.cartesian(label=f"Enzyme Family: {family_id}")
    axes.show = False # Hide the X/Y axis lines
    tre.draw(axes=axes, tip_labels=True, tip_labels_style={"font-size": 5})
    toyplot.svg.render(canvas, output_path)

# Example usage
draw_tree_with_background("./data/372.tree", "./trees/family_372.svg", "Family_372")